# 基于 MindSpore NLP 的 ERNIE 4.5 模型推理与应用

## 实验介绍

本实验主要介绍如何基于 MindSpore 2.7.0 AI 框架和 MindSpore NLP 0.5.1 套件，在 Ascend 800I/T A2 硬件环境下，实现 ERNIE 4.5 大语言模型的加载、推理及应用开发。

ERNIE 4.5 是百度开源的大规模模型系列，包含稠密（Dense）与混合专家（MoE）架构，在中文理解、多模态交互及长文本处理方面表现优异。本案例将演示如何利用 MindSpore 的 `AutoClass` 接口快速加载模型权重，并构建一个基于该模型的对话应用。

## 实验环境

本案例基于 **Ascend 800I/T A2** 硬件环境，软件环境如下：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10   | 2.7.0     | 0.5.1         |

### 安装依赖

首先，我们需要安装 MindNLP 及相关依赖库。如果环境中未安装，请执行以下命令：

In [ ]:
# 安装 MindSpore NLP
# !pip install mindnlp==0.5.1 -i https://pypi.tuna.tsinghua.edu.cn/simple
# 安装常用的文本处理库
# !pip install jieba
# !pip install sentencepiece

### 配置运行环境

引入必要的库，并设置 MindSpore 的运行模式。针对大模型推理，我们使用 Ascend 作为计算后端。

In [ ]:
import os
import time
import mindspore
from mindspore import context
import mindnlp

# 设置使用 Ascend 设备
# 默认使用 PYNATIVE_MODE 
context.set_context(device_target="Ascend")

print(f"MindSpore version: {mindspore.__version__}")
print("MindNLP version:", mindnlp.__version__)

## 数据准备

对于大模型推理任务，我们通常不需要像 CV NLP 等任务中那样下载大规模训练数据集。但在实际应用开发中，我们可能需要准备一些特定的 Prompt（提示词）或测试用例。

此处我们创建一个简单的测试数据集，模拟应用场景中的输入。

In [ ]:
# 模拟应用场景数据
test_cases = [
    "请简要介绍一下什么是混合专家模型（MoE）？",
    "写一首关于秋天丰收的七言绝句。",
    "请分析以下句子的情感倾向：'这家餐厅的服务真是太糟糕了，我再也不会来了。'",
    "使用Python写一个冒泡排序算法。"
]

print("测试用例准备完成。")

## 模型构建与加载

本章节将演示如何使用 MindSpore NLP 的 `Transformers` 接口加载 ERNIE 4.5 模型。

In [ ]:
# 加载分词器 (Tokenizer)
# 分词器负责将自然语言文本转换为模型可理解的 Token ID。

from mindnlp.transformers import AutoTokenizer
from mindnlp.transformers import AutoModelForCausalLM

MODEL_NAME = "baidu/ERNIE-4.5-0.3B-Base-PT"

print(f"正在加载分词器: {MODEL_NAME} ...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("分词器加载成功。")
except Exception as e:
    print(f"分词器加载失败，请检查网络或模型名称。错误信息: {e}")
    
# 加载模型 (Model)
# 在 Ascend 800I/T A2 上，为了节省显存并加速推理，我们推荐使用 float16 精度加载模型。

print(f"正在加载模型: {MODEL_NAME} ...")

# 加载模型权重
# mindspore_dtype=mindspore.float16 可以显著降低显存占用
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        mindspore_dtype=mindspore.float16
    )
    # 将模型设置为评估模式
    model.set_train(False)
    print("模型加载成功。")
except Exception as e:
    print(f"模型加载失败。错误信息: {e}")

## 应用开发：构建对话生成函数

为了方便进行多轮对话或特定任务推理，我们将模型的生成过程封装为一个函数。这类似于 ResNet 案例中的“验证”或“推理”步骤。

In [ ]:
def chat_with_ernie(query, history=[], max_length=2048, temperature=0.7, top_p=0.9):
    """
    基于 ERNIE 4.5 的对话生成函数
    
    Args:
        query (str): 用户输入的问题
        history (list): 对话历史
        max_length (int): 生成的最大长度
        temperature (float): 采样温度，控制生成的多样性
        top_p (float): 核采样阈值
    
    Returns:
        str: 模型生成的回答
    """
    # 1. 构建 Prompt
    # 说明：此示例针对 ERNIE 4.5 的 Base 预训练模型，直接对原始 query 做 tokenize，不使用额外 Chat Template。
    # 若使用的是已对话微调的 ERNIE 4.5 Chat 类模型，请先根据其官方 Chat Template 将 history 和 query 拼接为 prompt，再送入 tokenizer。
    inputs = tokenizer(query, return_tensors="ms")
    
    # 2. 生成配置
    # 注意：在 MindSpore 2.7 + MindSpore NLP 0.5.1 中，generate 接口用法与 Huggingface 类似
    outputs = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    
    # 3. 解码输出：仅解码生成的部分，避免误删或截断输入内容
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    return response.strip()

print("推理函数封装完成。")

## 实验结果展示

在本节中，我们将使用第3节准备的测试用例，对 ERNIE 4.5 模型进行实际的推理测试，展示其在不同领域的应用能力。

In [ ]:
# 知识问答任务
# 测试模型对专业知识的理解能力。

query_1 = test_cases[0] # 关于 MoE 的问题
print(f"Q: {query_1}")

start_time = time.time()
response_1 = chat_with_ernie(query_1)
end_time = time.time()

print(f"A: {response_1}")
print(f"推理耗时: {end_time - start_time:.2f} s")

In [ ]:
# 文学创作任务
# 测试模型的创意写作能力。

query_2 = test_cases[1] # 写诗
print(f"Q: {query_2}")
response_2 = chat_with_ernie(query_2)
print(f"A: \n{response_2}")

In [ ]:
# 情感分析任务
# 测试模型对自然语言的情绪理解能力。

query_3 = test_cases[2] # 情感分析
print(f"Q: {query_3}")
response_3 = chat_with_ernie(query_3)
print(f"A: \n{response_3}")

In [ ]:
# 逻辑与代码生成任务
# 测试模型的逻辑推理与代码能力。

query_4 = test_cases[3] # 写冒泡排序
print(f"Q: {query_4}")
response_4 = chat_with_ernie(query_4)
print(f"A: \n{response_4}")